**Last modified on**: 07/05/2026

**Author**: Onur Serçinoğlu

**Update Logs**:

07/05/2026: Initial version. Replaces the older `4_blast_msa_phylogeny.ipynb`.

**Credits**:

This Jupyter notebook accompanies the Phylogenetics lecture prepared for BENG451/BSB511 at Gebze Technical University. Key references:
- Saitou N, Nei M (1987). The neighbor-joining method: a new method for reconstructing phylogenetic trees. *Mol Biol Evol* 4:406–425.
- Sokal RR, Michener CD (1958). A statistical method for evaluating systematic relationships. *Univ Kansas Sci Bull* 38:1409.
- Felsenstein J (1985). Confidence limits on phylogenies: an approach using the bootstrap. *Evolution* 39:783–791.
- Yang Z (2014). *Molecular Evolution: A Statistical Approach.* Oxford University Press.

# Phylogenetic trees from a multiple sequence alignment

A phylogeny is a hypothesis about how a group of sequences (or species, or genes) descended from a common ancestor. From a multiple sequence alignment we can:

1. Compute pairwise **distances** between sequences.
2. Cluster them with a tree-building method — **UPGMA** (assumes a molecular clock) or **Neighbor-Joining** (does not).
3. **Root** the tree using an outgroup so that "parent" and "child" become defined.
4. Export the tree in **Newick** format for downstream tools.
5. Optionally assess clade support with **bootstrap** resampling.

In this notebook we build a globin tree end-to-end. We use the alignment produced by `5_2_msa.ipynb` (or fall back to a freshly-aligned `globin_set.fa`).

For interactive intuition on UPGMA and NJ, open the `phylo-interpreter.html` widget. For why distance choice matters, the `scoring-matrices.html` widget shows how PAM and BLOSUM differ.

In [ ]:
import os
import shutil
import subprocess

# ─── Adjustable paths ────────────────────────────────────────────────────────
WORK_DIR        = "phylo_work"
GLOBIN_SET      = "globin_set.fa"
MSA_INPUT       = "msa_work/globin_msa.aln"      # produced by 5_2_msa.ipynb (preferred)
MUSCLE_BIN      = shutil.which("muscle") or "muscle"
# ──────────────────────────────────────────────────────────────────────────────

os.makedirs(WORK_DIR, exist_ok=True)

# Prefer the alignment from notebook 5_2. If it is missing (e.g., student is
# running 5_3 first), fall back to aligning globin_set.fa here.
if os.path.exists(MSA_INPUT):
    ALN_PATH = MSA_INPUT
    print(f"Using MSA from 5_2:  {os.path.abspath(ALN_PATH)}")
else:
    ALN_PATH = os.path.join(WORK_DIR, "globin_msa_fallback.aln")
    print(f"5_2 alignment not found — running MUSCLE on {GLOBIN_SET} as fallback.")
    print("(Run 5_2_msa.ipynb first to use the canonical alignment.)")
    if not os.path.exists(ALN_PATH):
        result = subprocess.run([MUSCLE_BIN, "-align", GLOBIN_SET, "-output", ALN_PATH],
                                capture_output=True, text=True)
        if result.returncode != 0:
            raise SystemExit(f"MUSCLE fallback failed: {result.stderr}")
    print(f"Fallback alignment:  {os.path.abspath(ALN_PATH)}")

print(f"Working directory:   {os.path.abspath(WORK_DIR)}")

---

In [ ]:
import io
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from Bio import AlignIO, Phylo
from Bio.Phylo.TreeConstruction import DistanceCalculator, DistanceTreeConstructor

## 1. From alignment to a distance matrix

For each pair of aligned sequences $i, j$ we want a *distance* — small if they look similar, large if they look different. The simplest is **identity distance**:

$$d_{ij} = 1 - \frac{\text{aligned positions where } x_i = x_j}{\text{aligned positions where neither is a gap}}$$

A more biologically informed distance uses a substitution matrix (BLOSUM62, PAM250). Biopython's `DistanceCalculator` supports this directly. Internally, it maps each aligned column pair onto the matrix and normalises by the score of perfectly-matching columns:

$$d_{ij} = 1 - \frac{\sum_c s(x_{i,c}, x_{j,c})}{\max(\sum_c s(x_{i,c}, x_{i,c}),\ \sum_c s(x_{j,c}, x_{j,c}))}$$

Note: model name strings are **case-sensitive**: `'identity'`, `'blosum62'`, `'pam250'`.

In [ ]:
alignment = AlignIO.read(ALN_PATH, "fasta")
n_seq = len(alignment); n_col = alignment.get_alignment_length()
print(f"Loaded alignment: {n_seq} sequences × {n_col} columns from {ALN_PATH}\n")

# Build a short label for each tip so the trees are readable.
def short_label(record_id):
    # SwissProt-style: sp|P02185|MYG_PHYMC -> MYG_PHYMC
    parts = record_id.split("|")
    return parts[2] if len(parts) >= 3 else record_id

for r in alignment:
    label = short_label(r.id)
    r.id = label
    r.name = label
print("Tip labels:", [r.id for r in alignment])

In [ ]:
# Compute three distance matrices: identity, BLOSUM62, PAM250.
calc_id   = DistanceCalculator("identity")
calc_b62  = DistanceCalculator("blosum62")
calc_p250 = DistanceCalculator("pam250")

dm_id   = calc_id.get_distance(alignment)
dm_b62  = calc_b62.get_distance(alignment)
dm_p250 = calc_p250.get_distance(alignment)

def dm_to_array(dm):
    """Biopython's lower-triangular DistanceMatrix → symmetric numpy array."""
    n = len(dm.names)
    a = np.zeros((n, n))
    for i in range(n):
        for j in range(i + 1):
            a[i, j] = a[j, i] = dm[dm.names[i], dm.names[j]]
    return a, dm.names

a_id, names   = dm_to_array(dm_id)
a_b62, _      = dm_to_array(dm_b62)
a_p250, _     = dm_to_array(dm_p250)

print(f"Identity  range: {a_id.min():.3f} – {a_id.max():.3f}")
print(f"BLOSUM62  range: {a_b62.min():.3f} – {a_b62.max():.3f}")
print(f"PAM250    range: {a_p250.min():.3f} – {a_p250.max():.3f}")

In [ ]:
# Side-by-side heatmaps of the three distance matrices.
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for ax, (matrix, title) in zip(axes,
        [(a_id, "Identity"), (a_b62, "BLOSUM62"), (a_p250, "PAM250")]):
    im = ax.imshow(matrix, cmap="viridis", vmin=0, vmax=max(a_id.max(), a_b62.max(), a_p250.max()))
    ax.set_title(title)
    ax.set_xticks(range(len(names)))
    ax.set_yticks(range(len(names)))
    ax.set_xticklabels(names, rotation=90, fontsize=7)
    ax.set_yticklabels(names, fontsize=7)
    fig.colorbar(im, ax=ax, fraction=0.046)
fig.suptitle("Pairwise distance matrices under three models")
fig.tight_layout()
plt.show()

**Question 1:** Compare the BLOSUM62 and PAM250 panels. Where do they disagree most? PAM was estimated from highly similar (closely related) sequences and is then exponentiated to longer evolutionary distances; BLOSUM was estimated directly from blocks of more divergent sequences. What does that imply about which matrix is "right" for the leghemoglobin–vertebrate distance vs. the human α–human β distance?

*Answer hint:* PAM250 was extrapolated by matrix exponentiation (PAM1²⁵⁰), so it is least reliable at the largest evolutionary distances — exactly the ones we need for an outgroup like leghemoglobin. BLOSUM62 stays empirically grounded across distances and is the standard choice when the alignment spans the twilight zone.

## 2. UPGMA — the molecular-clock tree

UPGMA (Unweighted Pair Group Method with Arithmetic Mean) merges the two clusters with the smallest average inter-cluster distance, then updates the matrix with a weighted mean:

$$d_{(uv)k} = \frac{n_u\, d_{uk} + n_v\, d_{vk}}{n_u + n_v}$$

UPGMA assumes a **constant rate of evolution** — every leaf is equidistant from the root. The resulting tree is *ultrametric*. When the assumption holds, UPGMA recovers the right topology *and* the right divergence times. When it doesn't, it places fast-evolving lineages incorrectly.

For an interactive UPGMA walkthrough, see the `phylo-interpreter.html` widget.

In [ ]:
# Build the UPGMA tree from the BLOSUM62 distance matrix.
constructor = DistanceTreeConstructor()
tree_upgma = constructor.upgma(dm_b62)
print("UPGMA tree built. Total terminal nodes:", tree_upgma.count_terminals())

fig, ax = plt.subplots(figsize=(8, 6))
Phylo.draw(tree_upgma, axes=ax, do_show=False, branch_labels=lambda c: f"{c.branch_length:.2f}" if c.branch_length else "")
ax.set_title("UPGMA on BLOSUM62 distances")
plt.show()

## 3. Neighbor-Joining — without the clock assumption

Neighbor-Joining (NJ) does not assume a molecular clock. At each step it picks the pair $(i, j)$ that minimises a *corrected* distance:

$$Q_{ij} = (n - 2)\, d_{ij} - \sum_{k} d_{ik} - \sum_{k} d_{jk}$$

This penalises pairs that are merely close to *everything* rather than uniquely close to each other. NJ trees are not ultrametric; branch lengths reflect actual rates.

In [ ]:
tree_nj = constructor.nj(dm_b62)
print("NJ tree built. Total terminal nodes:", tree_nj.count_terminals())

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
Phylo.draw(tree_upgma, axes=axes[0], do_show=False)
axes[0].set_title("UPGMA")
Phylo.draw(tree_nj, axes=axes[1], do_show=False)
axes[1].set_title("Neighbor-Joining")
fig.tight_layout()
plt.show()

**Question 2:** Compare the two trees. Where does the topology differ? UPGMA forces every tip to be the same distance from the root; NJ does not. Looking at the NJ branch lengths, does the data look ultrametric? If not, which sequences are the most "out-of-clock" and what could explain that?

*Answer hint:* Plant leghemoglobin and the invertebrate globins (Glycera, Lumbricus, Chironomus) typically have very long terminal branches because they diverged from the vertebrate clade hundreds of millions of years ago. UPGMA pulls them in toward the centroid; NJ shows their true distance.

## 4. Rooting with an outgroup

Both algorithms above produce *unrooted* topologies (UPGMA happens to also place the root at the deepest split, but that root is only meaningful under the clock). To root meaningfully, we use an **outgroup** — a sequence we know diverged before everything else in the tree. Lupin leghemoglobin (`LGB2_LUPLU`) plays this role for the vertebrate + invertebrate animal globins: plants and animals split ~1.5 billion years ago, far before any internal split among animal globins.

If your input alignment came from `5_1_blast.ipynb` (which on a human β-globin query against SwissProt typically returns only *vertebrate* hemoglobins — see the discussion in 5_1 about plain BLAST's reach), leghemoglobin will not be there. The cell below detects that and falls back to **midpoint rooting**, which picks the root that minimises the maximum tip-to-root distance. Midpoint rooting assumes a near-clock — a reasonable assumption for a vertebrate-only β-globin tree, less so when invertebrate globins are mixed in.

Rooting with an outgroup *moves* the root but does not change relationships among the ingroup taxa.

In [ ]:
# Find the leghemoglobin tip and root the NJ tree on it. If the alignment
# came from 5_1's BLAST output (vertebrate-only set), leghemoglobin will be
# absent — fall back to midpoint rooting with a clear warning.
outgroup_label = "LGB2_LUPLU"
tip_names = [t.name for t in tree_nj.get_terminals()]
tree_nj_rooted = tree_nj

if outgroup_label in tip_names:
    tree_nj_rooted.root_with_outgroup({"name": outgroup_label})
    print(f"Rooted NJ tree on outgroup: {outgroup_label}")
else:
    print(f"⚠️  Outgroup {outgroup_label} not in alignment ({len(tip_names)} tips).")
    print("    This usually means 5_1's BLAST hit set was vertebrate-only.")
    print("    Falling back to midpoint rooting.")
    tree_nj_rooted.root_at_midpoint()

fig, ax = plt.subplots(figsize=(9, 6))
Phylo.draw(tree_nj_rooted, axes=ax, do_show=False)
ax.set_title(f"Rooted NJ tree (outgroup = {outgroup_label})")
plt.show()

**Question 3:** Why does adding an outgroup change *where* the root sits but not the relationships among the ingroup taxa? What if your dataset had no obvious outgroup — say, you were trying to root a tree of just vertebrate β-globins? What other rooting strategies could you use, and what assumptions would each one make?

*Answer hint:* The unrooted topology is uniquely determined by the distances; the root is just a chosen edge to "hang" the tree from. An outgroup picks the edge by knowledge of evolutionary history. *Midpoint rooting* picks the edge that minimises the maximum tip-to-root distance (assumes near-clock). *Molecular-clock rooting* fits a clock and picks the root that maximises likelihood. Each makes different assumptions about rate uniformity.

## 5. Newick export

Newick is a minimal, parenthesised text format that almost every phylogenetics tool consumes. We write the rooted NJ tree to disk and print it.

In [ ]:
newick_path = os.path.join(WORK_DIR, "globin_nj_rooted.nwk")
Phylo.write(tree_nj_rooted, newick_path, "newick")
print(f"Wrote {newick_path}")

with open(newick_path) as f:
    newick_text = f.read().strip()
print("\nNewick:")
print(newick_text)
print(f"\nTip: paste this into the `phylo-interpreter.html` widget on the course site to explore it interactively, or upload to https://itol.embl.de/ for a publication-quality figure.")

## 6. Bootstrap support (NJ only)

How confident are we that a particular clade is real? **Felsenstein's bootstrap** answers this empirically:

1. Resample the alignment columns *with replacement* to make a pseudo-alignment of the same width.
2. Recompute the distance matrix and rebuild the tree.
3. Repeat 100–1000 times; for each clade in the original tree, count how often the same clade appears in the bootstrap replicates.
4. The fraction is the *bootstrap support*, expressed as a percentage.

We bootstrap **only the NJ tree**, not UPGMA — UPGMA's ultrametric constraint produces near-invariant trees under resampling, so bootstrap on UPGMA is misleading.

In [ ]:
# Hand-rolled NJ bootstrap on BLOSUM62 distances. We keep this small (100
# replicates, 12 sequences × 184 columns) so it finishes in well under a minute.
import random
from Bio.Align import MultipleSeqAlignment
from Bio.SeqRecord import SeqRecord
from Bio.Seq import Seq

N_BOOT = 100
random.seed(42)

def resampled_alignment(aln, rng):
    L = aln.get_alignment_length()
    cols = [rng.randrange(L) for _ in range(L)]
    rows = []
    for r in aln:
        s = str(r.seq)
        rows.append(SeqRecord(Seq("".join(s[c] for c in cols)), id=r.id, name=r.id, description=""))
    return MultipleSeqAlignment(rows)

def clade_signature(tree):
    """Return the set of frozensets-of-leafnames for each non-trivial clade."""
    sigs = set()
    for clade in tree.get_nonterminals():
        leaves = frozenset(t.name for t in clade.get_terminals())
        if 1 < len(leaves) < tree.count_terminals():
            sigs.add(leaves)
    return sigs

orig_sigs = clade_signature(tree_nj)
clade_counts = {sig: 0 for sig in orig_sigs}

rng = random.Random(42)
for b in range(N_BOOT):
    boot_aln = resampled_alignment(alignment, rng)
    boot_dm  = calc_b62.get_distance(boot_aln)
    boot_tree = constructor.nj(boot_dm)
    boot_sigs = clade_signature(boot_tree)
    for sig in orig_sigs:
        if sig in boot_sigs:
            clade_counts[sig] += 1

print(f"Ran {N_BOOT} bootstrap replicates.\n")
print("Bootstrap support per clade (% of replicates):")
for sig, n in sorted(clade_counts.items(), key=lambda x: -x[1]):
    pct = 100 * n / N_BOOT
    members = sorted(sig)
    print(f"  {pct:5.1f}%  {{ {', '.join(members)} }}")

In [ ]:
# Annotate the original NJ tree with bootstrap support and re-render.
for clade in tree_nj_rooted.get_nonterminals():
    leaves = frozenset(t.name for t in clade.get_terminals())
    if leaves in clade_counts:
        clade.confidence = round(100 * clade_counts[leaves] / N_BOOT)

fig, ax = plt.subplots(figsize=(9, 6))
Phylo.draw(
    tree_nj_rooted,
    axes=ax,
    do_show=False,
    branch_labels=lambda c: f"{int(c.confidence)}" if c.confidence is not None else "",
)
ax.set_title(f"NJ tree, rooted on {outgroup_label}, with bootstrap support (n={N_BOOT})")
plt.show()

**Question 4:** A clade in your tree shows 67% bootstrap support. Is that "good"? In the literature, what is the conventional cutoff for considering a clade well-supported, and how does that change if your bootstrap was run on 100 replicates vs. 1000?

*Answer hint:* The classical rule of thumb is ≥ 70% support is "moderate", ≥ 95% is "strong", but that rule was calibrated for ML/parsimony bootstraps. For NJ on small alignments, 100 replicates have a noise floor of ±5 percentage points, so 67% is genuinely uncertain — you'd want 1000 replicates before reporting it as a meaningful number. Hillis & Bull (1993) is the standard reference for these calibrations.

## Resources

| Topic | Reference |
|-------|-----------|
| UPGMA | Sokal & Michener (1958). *Univ Kansas Sci Bull* 38:1409. |
| Neighbor-Joining | Saitou & Nei (1987). *Mol Biol Evol* 4:406. |
| Bootstrap | Felsenstein (1985). *Evolution* 39:783. |
| Bootstrap thresholds | Hillis & Bull (1993). *Syst Biol* 42:182. |
| Substitution matrices | Yang Z (2014). *Molecular Evolution: A Statistical Approach.* |
| Biopython Phylo | https://biopython.org/wiki/Phylo |
| Newick format | https://evolution.genetics.washington.edu/phylip/newicktree.html |
| iTOL (online tree viewer) | https://itol.embl.de/ |
| Course widget — UPGMA & NJ | `docs/phylo-interpreter.html` |
| Course widget — substitution matrices | `docs/scoring-matrices.html` |
| Previous notebook | `5_2_msa.ipynb` (produces the input alignment) |